# Baseline XLM-R: benchmark cấp đoạn A → B trên dữ liệu nguồn SemEval 2023

Notebook độc lập cho **Kaggle GPU**: đọc dữ liệu → chia train/validation → train XLM-R-base
với **tiêu đề + đoạn đích** → chọn ngưỡng trên validation → khóa mô hình/ngưỡng → dự đoán B
→ đánh giá tổng thể, từng nhãn, từng ngôn ngữ và đối chiếu scorer.

## Phạm vi cần hiểu đúng

- **Gói A:** 4.108 đoạn / 1.081 bài, DE/FR/IT/PL/RU. Khoảng 10% **nhóm bài** làm validation;
  số đoạn validation không nhất thiết đúng 10%. Các bản sao toàn bài cùng nhóm.
- **Gói B:** 1.768 đoạn / 466 bài, 9 ngôn ngữ; dùng làm **test cục bộ** theo protocol này.
  Không dùng second batch, không ghép A+B để train, không chọn ngưỡng trên B.
- **Hai gói hiện có là bản tái phát hành cho ST1 năm 2027 ở cấp đoạn.** Framing ST2 chính thức
  năm 2023 là **cấp bài báo**. README không xác nhận A/B tái tạo nguyên vẹn official split 2023.
  Điểm notebook này **không so trực tiếp với leaderboard 2023**. Không tự hợp nhãn đoạn thành gold cấp bài.
- Scorer nhúng là **ST1 năm 2027**, tính 14 frame + `None`; cũng báo riêng metrics 14 frame.
  Cả A/B hiện không có gold None. Không đặt mục tiêu điểm dựa trên các bảng tham chiếu chưa tương thích.
- EN/ES/GR/KA chưa xuất hiện trong **A của benchmark này**; English không phải surprise language
  của cuộc thi 2023 gốc. Cột seen/unseen được tính theo dữ liệu train thực tế.
- B đã có gold công khai và từng tham gia các thực nghiệm trước. Chỉ có thể giữ B ngoài lần fit này,
  không coi B là test ẩn độc lập chưa từng được xem. Bắt đầu từ **checkpoint tiền huấn luyện gốc**,
  không dùng model đã fine-tune trên A+B hoặc second batch.

**Cách chạy trên Kaggle:** Add Input dataset `sinhnguync/semeval-t9` và
[model XLM-R-base, version 1](https://www.kaggle.com/models/mohammedhamdan/xlm-roberta-base/Transformers/default/1),
chọn GPU trong Settings, rồi Run all. Ba đường dẫn dataset/model đã điền ở cell đầu.
Output lưu trong `/kaggle/working/SEMEVAL_T9_benchmarks`; không cần tải file Python/starter-kit.
Nếu cell dependencies cần cài thư viện còn thiếu, bật Internet cho bước cài đặt.
Sau khi chạy xong, Save Version để giữ output; tải model/ZIP từ mục Output/Files của Kaggle.

Đối chiếu: [trang task 2023](https://propaganda.math.unipd.it/semeval2023task3/),
[task overview 2023](https://aclanthology.org/2023.semeval-1.317/),
`README_ST1.md` trong release địa phương. Hướng dẫn gốc được giữ nguyên, các điểm khác biệt được làm rõ ở đây.


In [ ]:
from pathlib import Path

# TWO DATASET PATHS: point to each package, or a folder/ZIP containing that package.
TRAIN_DATA_PATH = Path("/kaggle/input/datasets/sinhnguync/semeval-t9/data/data/subtask1/SEMEVAL23_FRAMING_INITIAL_TRAIN-A")
TEST_DATA_PATH = Path("/kaggle/input/datasets/sinhnguync/semeval-t9/data/data/subtask1/SEMEVAL23_FRAMING_INITIAL_TRAIN-B")
# Add Input -> Models: mohammedhamdan/xlm-roberta-base, Transformers/default, version 1.
MODEL_NAME = "/kaggle/input/models/mohammedhamdan/xlm-roberta-base/transformers/default/1/xlm-roberta-base"

WORK_DIR = Path("/kaggle/working/semeval23_ab_work")
OUTPUT_ROOT = Path("/kaggle/working/SEMEVAL_T9_benchmarks")


In [ ]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Working directory:", WORK_DIR)


In [ ]:
# Install compatible dependencies in the active Kaggle kernel; restart if pip requests it.
import importlib.metadata
import importlib.util
import subprocess
import sys
from packaging.requirements import Requirement

required = {
    "numpy": ("numpy", "numpy>=1.24"), "pandas": ("pandas", "pandas>=2.0"),
    "sklearn": ("scikit-learn", "scikit-learn>=1.4,<2"), "torch": ("torch", "torch>=2.1"),
    "transformers": ("transformers", "transformers>=4.46,<5"),
    "accelerate": ("accelerate", "accelerate>=0.34,<2"),
    "sentencepiece": ("sentencepiece", "sentencepiece>=0.2"),
    "matplotlib": ("matplotlib", "matplotlib>=3.7"),
}
missing = []
for module, (distribution, spec) in required.items():
    try:
        valid = importlib.metadata.version(distribution) in Requirement(spec).specifier
    except importlib.metadata.PackageNotFoundError:
        valid = False
    if importlib.util.find_spec(module) is None or not valid:
        missing.append(spec)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies ready. Restart the kernel if an already-imported package was upgraded.")


In [ ]:
import importlib.metadata
import csv
import gc
import inspect
import json
import math
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string() if hasattr(value, 'to_string') else value)
from sklearn.metrics import (
    accuracy_score, f1_score, hamming_loss, precision_recall_fscore_support,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from torch.utils.data import Dataset
from transformers import (
    AutoConfig, AutoModelForSequenceClassification, AutoTokenizer, EarlyStoppingCallback,
    Trainer, TrainingArguments, set_seed,
)

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from transformers.trainer_utils import get_last_checkpoint


## 1. Cấu hình baseline

XLM-R-base + đầu classification chuẩn 14 logits, sigmoid, BCE với `sqrt(neg/pos)` chặn tối đa 4,
trọng số tính từ **subtrain A**. Input: `<s> tiêu đề </s></s> đoạn đích </s>`.
Giữ tối đa 48 token tiêu đề, 254 token đoạn; không có tiêu đề thì dùng đoạn đơn.
Tiêu đề cần đúng cấu trúc dòng đầu/dòng trống; đoạn đích là tiêu đề thì không lặp ngữ cảnh.

Tối đa 8 epoch, LR 2e-5, batch 8, tích lũy 2, chọn checkpoint theo validation micro-F1 tại 0,50.
Early stopping có thể kết thúc sớm. Sau đó mới tìm ngưỡng toàn cục/từng nhãn bằng validation A.
Đây là **một validation holdout theo nhóm**, không phải GroupKFold hay OOF.

`RUN_VERSION` phân biệt thí nghiệm. Mỗi run đã train được bảo vệ khỏi ghi đè. Muốn đọc model đã
hoàn thành: `RUN_TRAINING=False`, giữ cấu hình. Muốn tiếp tục run bị ngắt: đặt
`RESUME_FROM_CHECKPOINT="latest"` và giữ nguyên cấu hình/dữ liệu. Checkpoint cuối mới resume được;
run bị ngắt trước checkpoint đầu cần một RUN_VERSION mới.


In [ ]:
RUN_VERSION = "semeval23_ab_title_v1"
RUN_TRAINING = True
RESUME_FROM_CHECKPOINT = None  # None, "latest", or checkpoint folder INSIDE this run.

SEED = 42
VALIDATION_FRACTION = 0.10
MAX_LENGTH = 306
PARAGRAPH_MAX_TOKENS = 254
TITLE_MAX_TOKENS = 48
LEARNING_RATE = 2e-5
EPOCHS = 8
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 2
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.10
EARLY_STOPPING_PATIENCE = 3
POS_WEIGHT_MODE = "sqrt_cap4"
CHECKPOINT_THRESHOLD = 0.50
PRIMARY_METRIC = "official_micro_f1"
THRESHOLD_GRID = np.round(np.arange(0.15, 0.851, 0.05), 3)

if Path(RUN_VERSION).name != RUN_VERSION or "\\" in RUN_VERSION or RUN_VERSION in {"", ".", ".."}:
    raise ValueError("Use a plain folder name for RUN_VERSION")
RUN_DIR = OUTPUT_ROOT / "semeval23_AB_paragraph" / RUN_VERSION
MODEL_DIR = RUN_DIR / "model"
OUTPUT_DIR = RUN_DIR / "output"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOCK_PATH = MODEL_DIR / "benchmark_lock.json"
print("Run:", RUN_DIR)
print("CUDA required for practical full-size training; local CPU smoke is not a real score.")


## 2. Đọc duy nhất A và chia validation

Reader giữ nguyên đoạn/offset, kiểm tra khóa duy nhất, tên nhãn và giới hạn offset.
Nhãn nằm ở các cột TAB riêng; không split câu hoặc tự gán None cho đoạn chưa chú thích.
Hash toàn bài giúp giữ các bản sao cùng nhóm. B chưa được đọc ở bước này.


In [ ]:
"""A-only development and B-only final evaluation on the paragraph re-release.

Embedded in the standalone notebook; LABELS is set from package A before reading.
"""
import hashlib
import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


KEY_COLUMNS = ["doc_id", "begin_span", "end_span"]


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def prepare_dataset(path, cache_root):
    """Accept an existing directory or ZIP; reject path traversal and symlinks."""
    path = Path(path).expanduser().resolve()
    if path.is_dir():
        return path
    if not path.is_file() or path.suffix.lower() != ".zip":
        raise FileNotFoundError(f"Use a folder or ZIP path (not a sharing URL): {path}")
    digest = sha256_file(path)
    destination = Path(cache_root) / digest[:24]
    marker = destination / ".extraction_complete.json"
    if marker.is_file() and json.loads(marker.read_text(encoding="utf-8")).get("sha256") == digest:
        return destination
    with zipfile.ZipFile(path) as archive:
        for info in archive.infolist():
            name = info.filename.replace("\\", "/")
            member = Path(name)
            if (member.is_absolute() or ".." in member.parts or ":" in name or
                    (info.external_attr >> 16) & 0o170000 == 0o120000):
                raise ValueError("Unsafe ZIP member: " + info.filename)
        destination.mkdir(parents=True, exist_ok=True)
        archive.extractall(destination)
    marker.write_text(json.dumps({"sha256": digest}), encoding="utf-8")
    return destination


def locate_package(root, letter):
    """Resolve just ONE named package, never pool A/B or another release."""
    if letter not in {"A", "B"}:
        raise ValueError("Package must be A or B")
    root = Path(root).resolve()
    expected = "SEMEVAL23_FRAMING_INITIAL_TRAIN-" + letter
    candidates = [root] if root.name == expected else list(root.rglob(expected))
    candidates = sorted({path.resolve() for path in candidates if path.is_dir()
                         and (path / "labels.txt").is_file()
                         and any(path.glob("*/subtask-1-annotations-with-paragraphs.txt"))})
    if len(candidates) != 1:
        raise FileNotFoundError(f"Expected one {expected} under {root}; found {candidates}")
    return candidates[0]


def read_article_title(raw):
    """Use the documented first-line/blank-second-line format, never guess a title."""
    lines = raw.splitlines()
    if not lines or not lines[0].strip():
        return "", "missing_first_line"
    title = lines[0].strip()
    if title.casefold() in {"none", "null", "nan", "no title", "-"}:
        return "", "missing_title_marker"
    if len(lines) < 2 or lines[1].strip():
        return "", "ambiguous_layout"
    return title, "first_line"


def normalize_frames(values):
    values = [value.strip() for value in values if value.strip()]
    none_markers = {"none", "no framing", "-"}
    positives = [value for value in values if value.casefold() not in none_markers]
    negatives = [value for value in values if value.casefold() in none_markers]
    if negatives and positives:
        raise ValueError("None marker mixed with positive frames")
    if len(set(positives)) != len(positives) or set(positives) - set(LABELS):
        raise ValueError(f"Invalid or duplicate frames: {values}")
    return sorted(positives)


def read_benchmark_package(root, *, with_gold, source):
    """Do not even parse label fields for B inference; no fake all-zero gold."""
    root = Path(root).resolve()
    records, articles = [], {}
    paths = sorted(root.glob("*/subtask-1-annotations-with-paragraphs.txt"))
    if not paths:
        raise FileNotFoundError(f"Missing paragraph annotations: {root}")
    for path in paths:
        language = path.parent.name.upper()
        with path.open("r", encoding="utf-8-sig", newline="") as handle:
            for line_number, line in enumerate(handle, 1):
                fields = line.rstrip("\r\n").split("\t")
                if not any(fields):
                    continue
                if len(fields) < 4:
                    raise ValueError(f"{path}:{line_number}: missing key/text columns")
                doc_id, begin, end = fields[:3]
                if Path(doc_id).name != doc_id or "/" in doc_id or "\\" in doc_id:
                    raise ValueError(f"Invalid document ID: {doc_id}")
                begin, end = int(begin), int(end)
                paragraph = fields[-1]  # Preserve spaces; labels occupy separate TAB fields.
                if not paragraph.strip():
                    raise ValueError(f"Empty paragraph: {path}:{line_number}")
                raw_path = path.parent / "raw-documents" / doc_id
                if raw_path not in articles:
                    raw = raw_path.read_bytes().decode("utf-8-sig")  # Preserve CRLF for offsets.
                    article_hash = hashlib.sha256(re.sub(r"\s+", " ", raw).strip().encode("utf-8")).hexdigest()
                    articles[raw_path] = (raw, article_hash, *read_article_title(raw))
                raw, article_hash, title, title_status = articles[raw_path]
                if not 0 <= begin < end <= len(raw):
                    raise ValueError(f"Offset out of bounds: {doc_id} {begin}:{end}/{len(raw)}")
                raw_slice = raw[begin:end]
                alignment = "exact" if raw_slice == paragraph else (
                    "whitespace_equivalent" if re.sub(r"\s+", " ", raw_slice).strip() ==
                    re.sub(r"\s+", " ", paragraph).strip() else "supplied_paragraph_used")
                target_is_title = bool(title and begin == 0 and paragraph.strip().casefold() == title.casefold())
                record = {"doc_id": doc_id, "begin_span": begin, "end_span": end,
                          "paragraph": paragraph, "language": language, "source": source,
                          "article_path": str(raw_path), "article_sha256": article_hash,
                          "offset_alignment": alignment, "article_title": title,
                          "title_context": "" if target_is_title else title,
                          "title_status": "target_is_title" if target_is_title else title_status}
                if with_gold:
                    frames = normalize_frames(fields[3:-1])
                    record.update(frames=frames, **{label: int(label in frames) for label in LABELS})
                    record.update(label_count=len(frames), is_none=int(not frames))
                records.append(record)
    frame = pd.DataFrame(records)
    if frame.empty or frame.duplicated(KEY_COLUMNS).any():
        raise ValueError("Empty data or duplicate scorer keys")
    return frame


def attach_article_groups(frame):
    """Union both matching IDs and identical full articles, retaining every row."""
    frame = frame.copy().reset_index(drop=True)
    table = frame[["doc_id", "article_sha256"]].drop_duplicates().reset_index(drop=True)
    parent = list(range(len(table)))
    def find(index):
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index
    for column in ["doc_id", "article_sha256"]:
        seen = {}
        for index, value in enumerate(table[column]):
            if value in seen:
                a, b = find(index), find(seen[value])
                parent[max(a, b)] = min(a, b)
            else:
                seen[value] = index
    mapping = {(row.doc_id, row.article_sha256): f"article_group_{find(index):05d}"
               for index, row in enumerate(table.itertuples(index=False))}
    frame["group_id"] = [mapping[(row.doc_id, row.article_sha256)] for row in frame.itertuples(index=False)]
    return frame


def split_development(frame, *, seed, validation_fraction):
    if not 0 < validation_fraction < 1:
        raise ValueError("validation_fraction must be between 0 and 1")
    groups = frame.groupby("group_id", sort=True).language.agg(lambda values: "|".join(sorted(set(values))))
    counts = groups.value_counts()
    val_groups_count = int(np.ceil(len(groups) * validation_fraction))
    strata = groups if counts.min() >= 2 and len(counts) <= min(val_groups_count, len(groups)-val_groups_count) else None
    training_groups, validation_groups = train_test_split(groups.index.to_numpy(),
        test_size=validation_fraction, random_state=seed, stratify=strata)
    training = frame.loc[frame.group_id.isin(training_groups)].reset_index(drop=True)
    validation = frame.loc[frame.group_id.isin(validation_groups)].reset_index(drop=True)
    for column in ["doc_id", "article_sha256", "group_id"]:
        if set(training[column]) & set(validation[column]):
            raise AssertionError("Development split leakage: " + column)
    if len(training) + len(validation) != len(frame):
        raise AssertionError("Each A row must occur once")
    return training, validation


def frame_fingerprint(frame, *, with_gold):
    columns = KEY_COLUMNS + ["article_sha256", "paragraph", "title_context"]
    if with_gold:
        columns += LABELS
    rows = frame.sort_values(KEY_COLUMNS)[columns].to_dict("records")
    return hashlib.sha256(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()


def ordered_key_fingerprint(frame):
    """Prediction arrays and resumed samplers also depend on row order."""
    rows = frame[KEY_COLUMNS].to_dict("records")
    return hashlib.sha256(json.dumps(rows, sort_keys=True).encode("utf-8")).hexdigest()


def assert_no_test_overlap(development, test_inputs):
    for column in ["doc_id", "article_sha256"]:
        overlap = set(development[column]) & set(test_inputs[column])
        if overlap:
            raise ValueError(f"A/B overlap in {column}: {list(sorted(overlap))[:5]}")


def data_summary(frame):
    result = frame.groupby("language").agg(rows=("doc_id", "size"), documents=("doc_id", "nunique"),
        rows_with_title=("title_context", lambda values: int(values.ne("").sum())))
    if "is_none" in frame:
        result["gold_none"] = frame.groupby("language").is_none.sum()
    return result.reset_index()


In [ ]:
TRAIN_DATA_DIR = locate_package(prepare_dataset(TRAIN_DATA_PATH, WORK_DIR / "datasets_cache"), "A")
LABELS = sorted(line.strip() for line in (TRAIN_DATA_DIR / "labels.txt").read_text(
    encoding="utf-8-sig").splitlines() if line.strip())
if len(LABELS) != 14 or len(set(LABELS)) != 14:
    raise ValueError("Package A must define exactly 14 distinct frames")
development_df = attach_article_groups(read_benchmark_package(TRAIN_DATA_DIR, with_gold=True, source="A"))
train_df, val_df = split_development(development_df, seed=SEED, validation_fraction=VALIDATION_FRACTION)
training_languages = sorted(train_df.language.unique().tolist())
print(f"A: {len(development_df):,} rows / {development_df.doc_id.nunique():,} articles / "
      f"{development_df.group_id.nunique():,} content groups")
print(f"Subtrain: {len(train_df):,} rows | Validation: {len(val_df):,} rows")
display(pd.concat([data_summary(train_df).assign(split="train_A"),
                   data_summary(val_df).assign(split="validation_A")], ignore_index=True))
display(development_df.offset_alignment.value_counts().rename("rows").to_frame())


In [ ]:
"""Validation calibration and fixed-threshold paragraph benchmark evaluation.

This source is embedded in the standalone notebook after LABELS is defined.
The caller must pass validation predictions to calibrate_thresholds; test
evaluation deliberately exposes no option to fit or modify thresholds.
"""
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    hamming_loss,
    precision_recall_fscore_support,
)


def _benchmark_labels():
    labels = list(LABELS)
    if len(labels) != 14 or len(set(labels)) != 14 or "None" in labels:
        raise ValueError("LABELS must contain exactly 14 distinct frame names")
    return labels


def _binary_matrix(matrix, name):
    values = np.asarray(matrix)
    if values.ndim != 2 or values.shape[1] != len(_benchmark_labels()) or not len(values):
        raise ValueError(f"{name} must be a nonempty N x 14 matrix")
    try:
        valid = np.isfinite(values).all() and np.isin(values, [0, 1]).all()
    except TypeError as error:
        raise ValueError(f"{name} must contain finite binary values") from error
    if not valid:
        raise ValueError(f"{name} must contain finite binary values")
    return values.astype(np.int8, copy=False)


def _probability_matrix(probabilities):
    try:
        values = np.asarray(probabilities, dtype=np.float64)
    except (ValueError, TypeError) as error:
        raise ValueError("Probabilities must be numeric") from error
    if values.ndim != 2 or values.shape[1] != len(_benchmark_labels()) or not len(values):
        raise ValueError("Probabilities must be a nonempty N x 14 matrix")
    if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
        raise ValueError("Probabilities must be finite and between 0 and 1")
    return values


def _threshold_vector(thresholds):
    try:
        values = np.asarray(thresholds, dtype=np.float64)
    except (ValueError, TypeError) as error:
        raise ValueError("Thresholds must be numeric") from error
    if values.ndim == 0:
        values = np.full(len(_benchmark_labels()), float(values), dtype=np.float64)
    if values.shape != (len(_benchmark_labels()),):
        raise ValueError("Thresholds must be a scalar or a 14-element vector")
    if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
        raise ValueError("Thresholds must be finite and between 0 and 1")
    return values.copy()


def with_none_class(matrix):
    values = _binary_matrix(matrix, "Frame matrix")
    none = (values.sum(axis=1) == 0).astype(np.int8).reshape(-1, 1)
    return np.concatenate([values, none], axis=1)


def official_metrics_arrays(actual14, predicted14):
    actual = _binary_matrix(actual14, "Gold labels")
    predicted = _binary_matrix(predicted14, "Predictions")
    if actual.shape != predicted.shape:
        raise ValueError("Gold and prediction shapes differ")
    actual15, predicted15 = with_none_class(actual), with_none_class(predicted)
    return {
        "official_micro_f1": float(f1_score(actual15, predicted15, average="micro", zero_division=1)),
        "official_macro_f1": float(f1_score(actual15, predicted15, average="macro", zero_division=1)),
        "frames14_micro_f1": float(f1_score(actual, predicted, average="micro", zero_division=0)),
        "frames14_macro_f1": float(f1_score(actual, predicted, average="macro", zero_division=0)),
    }


def apply_thresholds(probabilities, thresholds):
    """Use independent >= decisions; no Top-1 fallback or implicit tuning."""
    values = _probability_matrix(probabilities)
    return (values >= _threshold_vector(thresholds)[None, :]).astype(np.int8)


def calibrate_thresholds(actual, probabilities, grid):
    """Fit on validation only: global grid, then <=3 coordinate passes.

    Changes require a strict official micro-F1 improvement. Labels with no
    positive validation examples retain the selected global threshold.
    """
    gold = _binary_matrix(actual, "Validation labels")
    probs = _probability_matrix(probabilities)
    if gold.shape != probs.shape:
        raise ValueError("Validation labels and probability shapes differ")
    candidates = np.asarray(grid, dtype=np.float64)
    if candidates.ndim != 1 or not len(candidates) or not np.isfinite(candidates).all():
        raise ValueError("Threshold grid must be a nonempty finite 1D sequence")
    if np.any((candidates < 0) | (candidates > 1)):
        raise ValueError("Threshold grid values must be between 0 and 1")
    candidates = np.unique(candidates)
    # On ties prefer the threshold closest to 0.50, then the smaller value.
    ordered = sorted(candidates.tolist(), key=lambda value: (abs(value - 0.5), value))
    best_global, best_score = None, -1.0
    global_scores = []
    for candidate in ordered:
        score = official_metrics_arrays(gold, apply_thresholds(probs, candidate))["official_micro_f1"]
        global_scores.append({"threshold": float(candidate), "official_micro_f1": score})
        if score > best_score + 1e-12:
            best_global, best_score = float(candidate), score
    thresholds = _threshold_vector(best_global)
    initial_score = best_score
    supported = gold.sum(axis=0) > 0
    trace, completed_passes = [], 0
    for pass_index in range(1, 4):
        changed = False
        for index, label in enumerate(_benchmark_labels()):
            if not supported[index]:
                continue
            chosen, candidate_score = thresholds[index], best_score
            for candidate in ordered:
                trial = thresholds.copy()
                trial[index] = candidate
                score = official_metrics_arrays(gold, apply_thresholds(probs, trial))["official_micro_f1"]
                if score > candidate_score + 1e-12:
                    chosen, candidate_score = float(candidate), score
            if candidate_score > best_score + 1e-12:
                previous = float(thresholds[index])
                thresholds[index], best_score = chosen, candidate_score
                trace.append({"pass": pass_index, "label": label,
                              "previous_threshold": previous, "threshold": float(chosen),
                              "official_micro_f1": float(best_score)})
                changed = True
        completed_passes = pass_index
        if not changed:
            break
    labels = _benchmark_labels()
    report = {
        "calibration_role": "validation_only_caller_must_enforce",
        "objective": "official_micro_f1_including_None",
        "global_threshold": best_global,
        "global_official_micro_f1": float(initial_score),
        "calibrated_official_micro_f1": float(best_score),
        "grid": candidates.tolist(), "coordinate_passes": completed_passes,
        "strict_improvement_tolerance": 1e-12, "top1_fallback": False,
        "unsupported_validation_labels": [label for label, valid in zip(labels, supported) if not valid],
        "label_thresholds": dict(zip(labels, thresholds.tolist())),
        "global_grid_scores": sorted(global_scores, key=lambda row: row["threshold"]),
        "coordinate_updates": trace,
    }
    return thresholds, report


def _frame_keys(frame):
    required = ["doc_id", "begin_span", "end_span"]
    if not all(column in frame.columns for column in required) or frame.empty:
        raise ValueError("A nonempty frame with doc_id/begin_span/end_span is required")
    keys = []
    for doc_id, begin, end in frame[required].itertuples(index=False, name=None):
        doc_id = str(doc_id)
        if not doc_id or any(character in doc_id for character in "\t\r\n"):
            raise ValueError("Article IDs must be nonempty and contain no TSV control characters")
        try:
            begin_int, end_int = int(begin), int(end)
            if float(begin) != begin_int or float(end) != end_int:
                raise ValueError("Offsets must be integers")
        except (ValueError, TypeError, OverflowError) as error:
            raise ValueError("Offsets must be finite integers") from error
        if begin_int < 0 or end_int <= begin_int:
            raise ValueError("Offsets must satisfy 0 <= begin_span < end_span")
        keys.append((doc_id, begin_int, end_int))
    if len(set(keys)) != len(keys):
        raise ValueError("Duplicate doc_id/begin_span/end_span keys")
    return keys


def write_predictions_tsv(frame, predictions, path):
    binary = _binary_matrix(predictions, "TSV predictions")
    keys = _frame_keys(frame)
    if len(keys) != len(binary):
        raise ValueError("Frame and prediction row counts differ")
    labels = _benchmark_labels()
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        for key, vector in zip(keys, binary):
            selected = [label for label, bit in zip(labels, vector) if bit]
            handle.write("\t".join([str(value) for value in key] + selected) + "\n")
    return path


def _gold_from_frame(frame):
    labels = _benchmark_labels()
    if all(label in frame.columns for label in labels):
        return _binary_matrix(frame[labels].to_numpy(), "Frame gold labels")
    if "frames" not in frame.columns:
        raise ValueError("Gold columns LABELS or a frames list column are required")
    matrix = np.zeros((len(frame), len(labels)), dtype=np.int8)
    lookup = {label: index for index, label in enumerate(labels)}
    for index, values in enumerate(frame["frames"]):
        if not isinstance(values, (list, tuple, set)):
            raise ValueError("Each frames value must be a list, tuple, or set of labels")
        if len(values) != len(set(values)) or set(values) - set(labels):
            raise ValueError("Unknown or duplicate gold frame names")
        for label in values:
            matrix[index, lookup[label]] = 1
    return matrix


def _full_metrics(actual, predicted, probabilities):
    summary = official_metrics_arrays(actual, predicted)
    actual15, predicted15 = with_none_class(actual), with_none_class(predicted)
    for prefix, gold, guess, zero in [("official", actual15, predicted15, 1),
                                      ("frames14", actual, predicted, 0)]:
        for average in ["micro", "macro"]:
            precision, recall, _, _ = precision_recall_fscore_support(
                gold, guess, average=average, zero_division=zero)
            summary[f"{prefix}_{average}_precision"] = float(precision)
            summary[f"{prefix}_{average}_recall"] = float(recall)
    supported = actual.sum(axis=0) > 0
    summary.update({
        "n_examples": int(len(actual)),
        "frames14_weighted_f1": float(f1_score(actual, predicted, average="weighted", zero_division=0)),
        "frames14_samples_f1": float(f1_score(actual, predicted, average="samples", zero_division=0)),
        "subset_accuracy": float(accuracy_score(actual, predicted)),
        "hamming_loss": float(hamming_loss(actual, predicted)),
        "gold_none_count": int((actual.sum(axis=1) == 0).sum()),
        "predicted_none_count": int((predicted.sum(axis=1) == 0).sum()),
        "average_precision_supported_label_count": int(supported.sum()),
        "frames14_average_precision_macro_supported": float(average_precision_score(
            actual[:, supported], probabilities[:, supported], average="macro")) if supported.any() else None,
        "frames14_average_precision_weighted_supported": float(average_precision_score(
            actual[:, supported], probabilities[:, supported], average="weighted")) if supported.any() else None,
        "official_zero_division": 1,
        "frames14_zero_division": 0,
    })
    return summary


def save_evaluation(frame, probabilities, thresholds, output_dir, prefix,
                    scorer_path, label_file, training_languages):
    """Evaluate fixed predictions and verify the local organizer scorer.

    Thresholds must have been frozen before this function is called. No part of
    this evaluation searches thresholds or changes a model.
    """
    _frame_keys(frame)
    probs = _probability_matrix(probabilities)
    actual = _gold_from_frame(frame)
    if actual.shape != probs.shape:
        raise ValueError("Evaluation frame and probability row counts differ")
    if "language" not in frame.columns or frame["language"].isna().any():
        raise ValueError("Evaluation frame requires nonmissing language values")
    values = _threshold_vector(thresholds)
    predicted = apply_thresholds(probs, values)
    output_dir = Path(output_dir)
    if not prefix or Path(prefix).name != prefix or any(char in prefix for char in "/\\"):
        raise ValueError("Output prefix must be a plain nonempty filename prefix")
    output_dir.mkdir(parents=True, exist_ok=True)
    gold_path = write_predictions_tsv(frame, actual, output_dir / f"{prefix}_gold.tsv")
    pred_path = write_predictions_tsv(frame, predicted, output_dir / f"{prefix}_predictions.tsv")
    np.save(output_dir / f"{prefix}_probabilities.npy", probs)
    labels = _benchmark_labels()
    summary = _full_metrics(actual, predicted, probs)
    summary.update({"threshold_mode": "fixed_per_label", "top1_fallback": False,
                    "label_thresholds": dict(zip(labels, values.tolist())),
                    "documents": int(frame.doc_id.nunique())})
    p, r, score, support = precision_recall_fscore_support(
        with_none_class(actual), with_none_class(predicted), average=None, zero_division=1)
    pd.DataFrame({"label": labels + ["None"], "precision": p, "recall": r,
                  "f1": score, "support": support.astype(int),
                  "predicted_count": with_none_class(predicted).sum(axis=0).astype(int),
                  "zero_division": 1}).to_csv(output_dir / f"{prefix}_per_label_metrics.csv", index=False)
    language_rows = []
    for language, positions in frame.groupby("language", sort=True).indices.items():
        positions = np.asarray(positions, dtype=int)
        language_rows.append({"language": language, "rows": len(positions),
            "documents": int(frame.iloc[positions].doc_id.nunique()),
            **_full_metrics(actual[positions], predicted[positions], probs[positions])})
    pd.DataFrame(language_rows).to_csv(output_dir / f"{prefix}_per_language_metrics.csv", index=False)
    if isinstance(training_languages, str):
        raise ValueError("training_languages must be a sequence, not a single string")
    seen_languages = set(training_languages)
    seen = frame.language.isin(seen_languages).to_numpy()
    group_rows = []
    for group, mask in [("seen_languages", seen), ("unseen_languages", ~seen)]:
        positions = np.flatnonzero(mask)
        if len(positions):
            group_rows.append({"group": group, "rows": len(positions),
                "documents": int(frame.iloc[positions].doc_id.nunique()),
                "languages": "|".join(sorted(set(frame.iloc[positions].language))),
                **_full_metrics(actual[positions], predicted[positions], probs[positions])})
    pd.DataFrame(group_rows).to_csv(output_dir / f"{prefix}_seen_unseen_language_metrics.csv", index=False)
    result = subprocess.run([sys.executable, str(scorer_path), "-g", str(gold_path),
        "-p", str(pred_path), "-f", str(label_file), "-o"], capture_output=True,
        text=True, encoding="utf-8", errors="replace")
    log = result.stdout + result.stderr
    (output_dir / f"{prefix}_official_scorer.txt").write_text(log, encoding="utf-8")
    if result.returncode:
        raise RuntimeError(f"Organizer scorer failed (exit {result.returncode}): {log}")
    score_lines = []
    for line in result.stdout.splitlines():
        fields = line.strip().split("\t")
        if len(fields) == 2:
            try:
                score_lines.append(tuple(map(float, fields)))
            except ValueError:
                continue
    if not score_lines:
        raise RuntimeError("Organizer scorer produced no parseable micro/macro result: " + log)
    micro, macro = score_lines[-1]
    np.testing.assert_allclose([micro, macro],
        [summary["official_micro_f1"], summary["official_macro_f1"]], atol=1e-10, rtol=0)
    summary["scorer_verified"] = True
    (output_dir / f"{prefix}_metrics.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    scalar_summary = {key: value for key, value in summary.items() if not isinstance(value, dict)}
    pd.DataFrame([scalar_summary]).to_csv(output_dir / f"{prefix}_metrics.csv", index=False)
    return summary


In [ ]:
def encode_paragraph_with_title(paragraph, title, tokenizer, *, max_length, paragraph_max_tokens, title_max_tokens):
    paragraph_ids = tokenizer.encode(
        paragraph, add_special_tokens=False, truncation=True, max_length=paragraph_max_tokens
    )
    title_ids = tokenizer.encode(
        title, add_special_tokens=False, truncation=True, max_length=title_max_tokens
    ) if title else []
    input_ids = (tokenizer.build_inputs_with_special_tokens(title_ids, paragraph_ids) if title_ids
                 else tokenizer.build_inputs_with_special_tokens(paragraph_ids))
    if len(input_ids) > max_length:
        raise ValueError(f"Encoded input has {len(input_ids)} tokens, above MAX_LENGTH={max_length}")
    return {"input_ids": input_ids, "attention_mask": [1] * len(input_ids)}

class ParagraphDataset(Dataset):
    def __init__(self, frame, tokenizer, *, with_labels, max_length=MAX_LENGTH,
                 paragraph_max_tokens=PARAGRAPH_MAX_TOKENS, title_max_tokens=TITLE_MAX_TOKENS):
        if "title_context" not in frame:
            raise ValueError("Missing title_context column; load data with read_st1_directory")
        required_length = paragraph_max_tokens + title_max_tokens + tokenizer.num_special_tokens_to_add(pair=True)
        if max_length < required_length:
            raise ValueError(f"MAX_LENGTH={max_length} cannot hold the configured budgets ({required_length})")
        texts = frame["paragraph"].fillna("").astype(str).tolist()
        titles = frame["title_context"].fillna("").astype(str).tolist()
        self.inputs = [
            encode_paragraph_with_title(text, title, tokenizer, max_length=max_length,
                                        paragraph_max_tokens=paragraph_max_tokens,
                                        title_max_tokens=title_max_tokens)
            for text, title in zip(texts, titles)
        ]
        self.labels = frame[LABELS].to_numpy(dtype=np.float32) if with_labels else None

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        # The collator pops labels, so return a copy of the cached input dictionary.
        item = dict(self.inputs[index])
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[index], dtype=torch.float32)
        return item

class MultilabelCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, examples):
        have_labels = "labels" in examples[0]
        labels = torch.stack([example.pop("labels") for example in examples]) if have_labels else None
        batch = self.tokenizer.pad(
            examples, padding=True,
            pad_to_multiple_of=8 if torch.cuda.is_available() else None,
            return_tensors="pt",
        )
        if labels is not None:
            batch["labels"] = labels
        return batch

class WeightedMultilabelTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = F.binary_cross_entropy_with_logits(
            outputs.logits, labels.float(), pos_weight=self.pos_weight.to(outputs.logits.device)
        )
        return (loss, outputs) if return_outputs else loss

def class_weights(frame, mode=POS_WEIGHT_MODE, *, show=False):
    if mode == "none":
        weights = np.ones(len(LABELS), dtype=np.float32)
    elif mode in {"sqrt_cap2", "sqrt_cap4"}:
        positives = frame[LABELS].sum().to_numpy(dtype=np.float32)
        ratio = (len(frame) - positives) / np.maximum(positives, 1.0)
        cap = 2.0 if mode == "sqrt_cap2" else 4.0
        weights = np.clip(np.sqrt(ratio), 1.0, cap).astype(np.float32)
    else:
        raise ValueError("Weight mode must be none, sqrt_cap2, or sqrt_cap4")
    if show:
        display(pd.DataFrame({"label": LABELS, "pos_weight": weights}))
    return torch.tensor(weights, dtype=torch.float32)

def audit_token_lengths(frame, tokenizer, max_length=MAX_LENGTH,
                        paragraph_max_tokens=PARAGRAPH_MAX_TOKENS,
                        title_max_tokens=TITLE_MAX_TOKENS):
    texts = frame["paragraph"].fillna("").astype(str).tolist()
    titles = frame["title_context"].fillna("").astype(str).tolist()
    paragraph_lengths = [len(tokenizer.encode(t, add_special_tokens=False, verbose=False)) for t in texts]
    title_lengths = [len(tokenizer.encode(t, add_special_tokens=False, verbose=False)) if t else 0 for t in titles]
    dataset = ParagraphDataset(
        frame, tokenizer, with_labels=False, max_length=max_length,
        paragraph_max_tokens=paragraph_max_tokens, title_max_tokens=title_max_tokens,
    )
    model_lengths = [len(item["input_ids"]) for item in dataset]
    audit = frame[["doc_id", "begin_span", "end_span", "language", "title_status"]].copy()
    audit["paragraph_tokens"] = paragraph_lengths
    audit["title_tokens"] = title_lengths
    audit["model_input_tokens"] = model_lengths
    audit["paragraph_truncated"] = audit["paragraph_tokens"] > paragraph_max_tokens
    audit["title_truncated"] = audit["title_tokens"] > title_max_tokens
    summary = {
        "rows": len(audit), "max_length": max_length,
        "paragraph_max_tokens": paragraph_max_tokens, "title_max_tokens": title_max_tokens,
        "title_context_fraction": float((audit["title_tokens"] > 0).mean()),
        "paragraph_truncated_fraction": float(audit["paragraph_truncated"].mean()),
        "title_truncated_fraction": float(audit["title_truncated"].mean()),
        "model_input_p50": float(np.quantile(model_lengths, .50)),
        "model_input_p90": float(np.quantile(model_lengths, .90)),
        "model_input_p95": float(np.quantile(model_lengths, .95)),
        "model_input_max": int(max(model_lengths)),
    }
    audit.to_csv(OUTPUT_DIR / "token_length_audit.csv", index=False)
    display(pd.Series(summary, name="value").to_frame())
    display(audit.groupby("language")["paragraph_truncated"].mean().rename("fraction_paragraph_truncated").to_frame())
    return summary

def trainer_kwargs_with_tokenizer(kwargs, tokenizer):
    key = "processing_class" if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer"
    kwargs[key] = tokenizer
    return kwargs

def training_arguments(**kwargs):
    key = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"
    kwargs[key] = "epoch"
    return TrainingArguments(**kwargs)


def checkpoint_metrics(eval_prediction):
    logits = eval_prediction.predictions[0] if isinstance(eval_prediction.predictions, tuple) else eval_prediction.predictions
    probabilities = torch.sigmoid(torch.as_tensor(logits)).cpu().numpy()
    return official_metrics_arrays(np.asarray(eval_prediction.label_ids, dtype=np.int8),
                                   apply_thresholds(probabilities, CHECKPOINT_THRESHOLD))


## 3. Train, lưu checkpoint và nạp lại mô hình

Kiểm tra checkpoint nguồn không phải sequence classifier đã fine-tune. Kiểm tra này bắt được các
checkpoint tác vụ thông thường nhưng không chứng minh lịch sử mọi file trọng số; cần tự bảo đảm
đường dẫn local là model tiền huấn luyện gốc. Các cảnh báo khởi tạo classifier mới là bình thường.

Manifest cấu hình/dữ liệu được lưu trước train. Khác dữ liệu, split hoặc siêu tham số thì cần run mới.
Model tốt nhất trên validation được lưu ở `model/`; checkpoint optimizer ở `checkpoints/` dùng resume.


In [ ]:
# Exact organizer scorer and implementation fingerprint embedded below.
SCORER_SOURCE = 'import logging.handlers\nimport argparse\nimport sys\nimport os\nfrom sklearn.metrics import f1_score\nfrom sklearn.preprocessing import MultiLabelBinarizer\nimport sklearn\n\n"""\nScorer for SemEval 2027 subtask 1.\nSince it is a multilabel multiclass classification problem, we compute the micro F1 over all the classes.\nOne row of the prediction file is the following:\narticle_id|TAB|begin_span|TAB|end_span|TAB|frame1|TAB|frame2|TAB|...|TAB|frameN\n\nwhere article_id is the numerical code in the name of the file with the input article, |TAB| is the tab character,\nbegin_span is the beginning of the span of the paragraph, end span is the end span of the paragraph and frame1|TAB|frame2|TAB|...|TAB|frameN is a tab-separated list of frames (the list could be empty).\nFor example:\n\narticle_id_1\t0\t56\tQuality_of_life\narticle_id_1\t62\t381\tQuality_of_life\narticle_id_2\t387\t780\tPolicy_prescription_and_evaluation\tQuality_of_life\narticle_id_2\t784\t1113\tCapacity_and_resources\narticle_id_2\t1117\t1595\tEconomic\tFairness_and_equality\narticle_id_2\t1599\t2018\tEconomic\tQuality_of_life\narticle_id_2\t2022\t2349\tEconomic\tFairness_and_equality\n\nAn empty frame list (no framing detected) is scored against an explicit "None" class.\n"""\n\nNONE_LABEL = "None"\n\nlogger = logging.getLogger("subtask1_scorer")\nch = logging.StreamHandler(sys.stdout)\nch.setLevel(logging.INFO)\nformatter = logging.Formatter(\'%(asctime)s - %(levelname)s - %(message)s\')\nch.setFormatter(formatter)\nlogger.setLevel(logging.INFO)\n\n\ndef read_frame_list_from_file(file_full_name):\n  """\n  Read the list of frame names from a file, one per line.\n  """\n  with open(file_full_name, encoding=\'utf-8\') as f:\n    return [ line.rstrip() for line in f.readlines() ]\n\n\ndef _read_csv_input_file(file_full_name):\n  """\n  Read a csv file with two columns TAB separated:\n   - first column is the id of the example\n   - second column is the begin span\n   - third column is the end span\n   - remaining columns is the list of labels of the example\n  """\n  d = list()\n  with open(file_full_name, encoding=\'utf-8\') as f:\n    for line in f.readlines():\n      line = line.rstrip(\'\\n\')\n      if not line:\n        continue\n      line_data = line.split("\\t")\n      if len(line_data) < 3:\n        logger.error(\'ERROR: each line of the file is supposed to have at least a doc_id, a begin span, and an end span, found inconsistencies in line \' + line)\n        sys.exit(1)\n      a = dict()\n      a["doc_id"] = line_data.pop(0)\n      a["begin_span"] = line_data.pop(0)\n      a["end_span"] = line_data.pop(0)\n      a["frames"] = line_data\n      d.append(a)\n  return d\n\n\ndef _labels_correct(labels, CLASSES, debug=False):\n  """\n  Make sure all the labels correspond to strings in the CLASSES array\n  :param labels: a list of dictionaries\n  :param CLASSES: a list of allowed labels\n  """\n  if debug:\n    s=""\n    for entry in labels:\n     for l in entry["frames"]:\n         if l not in CLASSES:\n             s += l + " "\n    return s\n  else:\n    for entry in labels:\n      for l in entry["frames"]:\n        if l not in CLASSES:\n          return False\n  return True\n\n\ndef _key(entry):\n  """\n  Unique key for a row: doc_id and span.\n  """\n  return entry["doc_id"]+"_"+entry["begin_span"]+"_"+entry["end_span"]\n\n\ndef _dedupe_keeping_last(labels, source_name):\n  """\n  Keep the last occurrence of each duplicate row, warning about the rest.\n  """\n  deduped = {}\n  for entry in labels:\n    key = _key(entry)\n    if key in deduped:\n      logger.warning(\'WARNING: duplicate entry for "%s" found in the %s file; only the last occurrence will be used for scoring.\'\n                     % (key, source_name))\n    deduped[key] = entry\n  return list(deduped.values())\n\n\ndef _correct_number_of_examples(pred_labels, gold_labels):\n  """\n  Make sure that the number of predictions is exactly the same as the gold labels\n  """\n  return len(pred_labels)==len(gold_labels)\n\n\ndef _correct_id_list(pred_labels, gold_labels, debug=False):\n  """\n  Check that the keys of pred_labels match the gold file.\n  """\n  pred_ids = set(_key(entry) for entry in pred_labels)\n  gold_ids = set(_key(entry) for entry in gold_labels)\n  if debug:\n    return ", ".join(pred_ids.symmetric_difference(gold_ids))\n  return pred_ids == gold_ids\n\n\ndef _extract_matching_lists(pred_labels, gold_labels):\n  """\n  Extract the list of values from the two list of dictionaries ensuring that elements with the same key are in the same position.\n  """\n  pred_by_id = {_key(entry): entry["frames"] for entry in pred_labels}\n  pred_values, gold_values = ([],[])\n  for entry in gold_labels:\n    gold_values.append(entry["frames"])\n    pred_values.append(pred_by_id[_key(entry)])\n  return pred_values, gold_values\n\n\ndef _to_scoring_labels(frame_lists):\n  """\n  Map an empty frame list to the None label.\n  """\n  return [labels if labels else [NONE_LABEL] for labels in frame_lists]\n\n\ndef correct_format(pred_labels, gold_labels, CLASSES):\n  """\n  Check whether the format of the prediction file is correct.\n  The number of checks that can be performed depends on the availability of the gold labels.\n  Runs every check and reports all problems found, rather than stopping at the first one.\n  """\n  is_correct = True\n\n  if not _labels_correct(pred_labels, CLASSES):\n    logger.error(\'ERROR: the following labels in the prediction file are not valid: {}.\'\n                 .format(_labels_correct(pred_labels, CLASSES, True)))\n    is_correct = False\n\n  if gold_labels: # we can do further checks if the gold_labels are available\n    if not _correct_number_of_examples(pred_labels, gold_labels):\n      logger.error(\'The number of predictions (%d) is not the expected one (%d)\'\n                   %(len(pred_labels), len(gold_labels)))\n      is_correct = False\n\n    if not _correct_id_list(pred_labels, gold_labels):\n      logger.error(\'The list of articles ids+spans is not correct. The following entries are not in the gold file or absent from the prediction file: %s\'\n                   %(_correct_id_list(pred_labels, gold_labels, True)))\n      is_correct = False\n\n  return is_correct\n\n\ndef evaluate(pred_labels, gold_labels, CLASSES):\n  """\n    Evaluates the predicted classes w.r.t. a gold file.\n    Metrics are: multilabel macro_f1 and micro_f1.\n    :param pred_labels: a dictionary with predictions,\n    :param gold_labels: a dictionary with gold labels.\n  """\n  pred_values, gold_values = _extract_matching_lists(pred_labels, gold_labels)\n  pred_values = _to_scoring_labels(pred_values)\n  gold_values = _to_scoring_labels(gold_values)\n\n  mlb = MultiLabelBinarizer()\n  mlb.fit([CLASSES + [NONE_LABEL]])\n  gold_values = mlb.transform(gold_values)\n  pred_values = mlb.transform(pred_values)\n\n  macro_f1 = f1_score(gold_values, pred_values, average="macro", zero_division=1)\n  micro_f1 = f1_score(gold_values, pred_values, average="micro", zero_division=1)\n  return macro_f1, micro_f1\n\n\nif __name__ == \'__main__\':\n\n  parser = argparse.ArgumentParser()\n  parser.add_argument("--gold_file_path", \'-g\', type=str, required=False, help="Paths to the file with gold annotations.")\n  parser.add_argument("--pred_file_path", \'-p\', type=str, required=True, help="Path to the file with predictions")\n  parser.add_argument("--frame_file_path", \'-f\', type=str, required=True, help="Path to the file with the names of the frames")\n  parser.add_argument("--log_to_file", "-l", action=\'store_true\', default=False,\n                      help="Set flag if you want to log the execution file. The log will be appended to <pred_file>.log")\n  parser.add_argument(\'--output-for-script\', "-o", dest=\'output_for_script\', required=False, action=\'store_true\',\n                      default=False, help="Prints the output in a format easy to parse for a script")\n  args = parser.parse_args()\n\n  output_for_script = bool(args.output_for_script)\n  if not output_for_script:\n    logger.addHandler(ch)\n\n  CLASSES = read_frame_list_from_file(args.frame_file_path)\n\n  pred_file = args.pred_file_path\n  if args.gold_file_path:\n    gold_file = args.gold_file_path\n  else:\n    gold_file = None\n  if args.log_to_file:\n    output_log_file = pred_file + ".log"\n    logger.info("Logging execution to file " + output_log_file)\n    fileLogger = logging.FileHandler(output_log_file)\n    fileLogger.setLevel(logging.DEBUG)\n    fileLogger.setFormatter(formatter)\n    logger.addHandler(fileLogger)\n    logger.setLevel(logging.DEBUG) #\n\n  if args.log_to_file:\n    logger.info(\'Reading predictions file\')\n  else:\n    logger.info(\'Reading predictions file {}\'.format(args.pred_file_path))\n  if gold_file:\n    if args.log_to_file:\n      logger.info(\'Reading gold file\')\n    else:\n      logger.info("Reading gold predictions from file {}".format(args.gold_file_path))\n  else:\n    logger.info(\'No gold file provided\')\n\n  pred_labels = _read_csv_input_file(pred_file)\n  gold_labels = _read_csv_input_file(gold_file) if gold_file else None\n\n  pred_labels = _dedupe_keeping_last(pred_labels, "predictions")\n  if gold_labels:\n    gold_labels = _dedupe_keeping_last(gold_labels, "gold")\n\n  if correct_format(pred_labels, gold_labels, CLASSES):\n    logger.info(\'Prediction file format is correct\')\n    if gold_labels:\n      macro_f1, micro_f1 = evaluate(pred_labels, gold_labels, CLASSES)\n      logger.info("micro-F1={:.5f}\\tmacro-F1={:.5f}".format(micro_f1, macro_f1))\n      if output_for_script:\n        print("{}\\t{}".format(micro_f1, macro_f1))\n'
IMPLEMENTATION_SHA256 = 'bc9d7ca414ec2c7efb64d479ff1b27c821df1bf9d9e76007479c5eed7f6a0e37'

def protocol_specification():
    return {
        "protocol": "local_paragraph_A_train_B_test_v1",
        "implementation_sha256": IMPLEMENTATION_SHA256,
        "base_model": str(MODEL_NAME), "labels": LABELS,
        "input_format": "title_then_target_xlmr_pair", "max_length": MAX_LENGTH,
        "paragraph_max_tokens": PARAGRAPH_MAX_TOKENS, "title_max_tokens": TITLE_MAX_TOKENS,
        "seed": SEED, "validation_fraction_groups": VALIDATION_FRACTION,
        "development_sha256": frame_fingerprint(development_df, with_gold=True),
        "subtrain_sha256": frame_fingerprint(train_df, with_gold=True),
        "validation_sha256": frame_fingerprint(val_df, with_gold=True),
        "subtrain_ordered_keys_sha256": ordered_key_fingerprint(train_df),
        "validation_ordered_keys_sha256": ordered_key_fingerprint(val_df),
        "train_rows": len(train_df), "validation_rows": len(val_df),
        "train_languages": training_languages,
        "learning_rate": LEARNING_RATE, "epochs": EPOCHS,
        "train_batch_size": TRAIN_BATCH_SIZE, "eval_batch_size": EVAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "weight_decay": WEIGHT_DECAY, "warmup_fraction": WARMUP_FRACTION,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "pos_weight_mode": POS_WEIGHT_MODE, "checkpoint_threshold": CHECKPOINT_THRESHOLD,
        "threshold_grid": THRESHOLD_GRID.tolist(),
        "scorer_sha256": hashlib.sha256(SCORER_SOURCE.encode("utf-8")).hexdigest(),
        "versions": {name: importlib.metadata.version(name) for name in
                     ["torch", "transformers", "accelerate", "scikit-learn", "numpy", "pandas"]},
    }


def initialize_run():
    protocol = protocol_specification()
    protocol_path = RUN_DIR / "protocol.json"
    resume = None
    if protocol_path.is_file():
        previous = json.loads(protocol_path.read_text(encoding="utf-8"))
        if previous != protocol:
            changed = [key for key in protocol if previous.get(key) != protocol[key]]
            raise ValueError(f"Run protocol changed ({changed}). Set a NEW RUN_VERSION.")
        if RUN_TRAINING:
            if LOCK_PATH.exists() or RESUME_FROM_CHECKPOINT is None:
                raise FileExistsError("Existing run: use a NEW RUN_VERSION, load with RUN_TRAINING=False, "
                                      "or resume an unfinished run explicitly.")
            resume = get_last_checkpoint(str(CHECKPOINT_DIR)) if str(RESUME_FROM_CHECKPOINT) == "latest" else str(RESUME_FROM_CHECKPOINT)
            if not resume or not Path(resume).is_dir() or CHECKPOINT_DIR.resolve() not in Path(resume).resolve().parents:
                raise FileNotFoundError("No valid resume checkpoint inside this run")
        elif not LOCK_PATH.is_file():
            raise FileNotFoundError("No completed locked model. Resume training first.")
    else:
        if not RUN_TRAINING or RESUME_FROM_CHECKPOINT is not None:
            raise FileNotFoundError("Run does not exist: start with RUN_TRAINING=True and no resume.")
        if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
            raise FileExistsError("Run folder contains unrecognized files. Choose another RUN_VERSION.")
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        protocol_path.write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, frame in [("train_A", train_df), ("validation_A", val_df)]:
        frame.to_csv(OUTPUT_DIR / f"{name}_manifest.csv", index=False)
    data_summary(development_df).to_csv(OUTPUT_DIR / "package_A_summary.csv", index=False)
    development_df[["doc_id", "article_sha256", "title_status", "article_title"]].drop_duplicates().to_csv(
        OUTPUT_DIR / "title_extraction_A.csv", index=False)
    (OUTPUT_DIR / "labels.txt").write_text("\n".join(LABELS) + "\n", encoding="utf-8")
    (OUTPUT_DIR / "scorer-subtask-1.py").write_text(SCORER_SOURCE, encoding="utf-8")
    return protocol, resume


def validate_pretrained_source():
    if str(MODEL_NAME).startswith("/kaggle/input/") and not (Path(MODEL_NAME) / "config.json").is_file():
        raise FileNotFoundError("Attach the Kaggle XLM-R model via Add Input -> Models. "
                                "MODEL_NAME must point to its folder containing config.json: " + str(MODEL_NAME))
    config = AutoConfig.from_pretrained(MODEL_NAME)
    if config.model_type != "xlm-roberta":
        raise ValueError("This baseline expects XLM-R; change the implementation for another encoder.")
    if any("Classification" in name for name in (config.architectures or [])):
        raise ValueError("Use an UNTUNED pretrained XLM-R checkpoint, not an existing task classifier.")
    local = Path(MODEL_NAME)
    if local.is_dir() and any((local / name).exists() for name in
                             ["task_metadata.json", "benchmark_lock.json", "custom_model.pt"]):
        raise ValueError("Task checkpoint detected. Start from the pretrained encoder.")
    maximum = config.max_position_embeddings - config.pad_token_id - 1
    if MAX_LENGTH > maximum:
        raise ValueError(f"MAX_LENGTH={MAX_LENGTH} exceeds encoder capacity {maximum}")


def fit_baseline(resume):
    validate_pretrained_source()
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    token_summary = audit_token_lengths(train_df, tokenizer)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS), problem_type="multi_label_classification",
        id2label=dict(enumerate(LABELS)), label2id={label: index for index, label in enumerate(LABELS)})
    training_set = ParagraphDataset(train_df, tokenizer, with_labels=True)
    validation_set = ParagraphDataset(val_df, tokenizer, with_labels=True)
    weights = class_weights(train_df, show=True)
    pd.DataFrame({"label": LABELS, "train_positives": train_df[LABELS].sum().to_numpy(),
                  "pos_weight": weights.numpy()}).to_csv(OUTPUT_DIR / "train_class_weights.csv", index=False)
    steps_per_epoch = math.ceil(len(training_set) / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS))
    bf16_ok = torch.cuda.is_available() and bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    args = training_arguments(
        output_dir=str(CHECKPOINT_DIR), learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY, lr_scheduler_type="linear",
        warmup_steps=max(1, int(steps_per_epoch * EPOCHS * WARMUP_FRACTION)),
        per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        save_strategy="epoch", save_total_limit=2, logging_strategy="steps", logging_steps=25,
        load_best_model_at_end=True, metric_for_best_model=PRIMARY_METRIC, greater_is_better=True,
        bf16=bf16_ok, fp16=torch.cuda.is_available() and not bf16_ok,
        report_to="none", seed=SEED, data_seed=SEED, dataloader_num_workers=0,
    )
    kwargs = trainer_kwargs_with_tokenizer(dict(model=model, args=args, train_dataset=training_set,
        eval_dataset=validation_set, data_collator=MultilabelCollator(tokenizer),
        pos_weight=weights, compute_metrics=checkpoint_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)]), tokenizer)
    trainer = WeightedMultilabelTrainer(**kwargs)
    if hasattr(trainer, "model_accepts_loss_kwargs"):
        trainer.model_accepts_loss_kwargs = False
    result = trainer.train(resume_from_checkpoint=resume)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(MODEL_DIR))
    tokenizer.save_pretrained(str(MODEL_DIR))
    pd.DataFrame(trainer.state.log_history).to_csv(OUTPUT_DIR / "training_log.csv", index=False)
    summary = {"best_checkpoint": trainer.state.best_model_checkpoint,
               "best_validation_f1_at_0_5": trainer.state.best_metric,
               "completed_epochs": trainer.state.epoch, "training": result.metrics,
               "token_summary": token_summary}
    (OUTPUT_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    del trainer, model, training_set, validation_set
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def predict_saved_model(frame):
    """Reload best saved model and predict without labels or a test loss."""
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
    expected = [model.config.id2label[index] for index in range(len(LABELS))]
    if expected != LABELS:
        raise ValueError("Saved label order mismatch")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    dataset = ParagraphDataset(frame, tokenizer, with_labels=False)
    bf16_ok = torch.cuda.is_available() and bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    args = TrainingArguments(output_dir=str(OUTPUT_DIR / "inference_runtime"),
        per_device_eval_batch_size=EVAL_BATCH_SIZE, report_to="none", dataloader_num_workers=0,
        bf16=bf16_ok, fp16=torch.cuda.is_available() and not bf16_ok)
    predictor = Trainer(**trainer_kwargs_with_tokenizer(dict(model=model, args=args,
                        data_collator=MultilabelCollator(tokenizer)), tokenizer))
    result = predictor.predict(dataset)
    logits = result.predictions[0] if isinstance(result.predictions, tuple) else result.predictions
    probabilities = torch.sigmoid(torch.as_tensor(logits)).cpu().numpy()
    del predictor, model, dataset, result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if probabilities.shape != (len(frame), len(LABELS)) or not np.isfinite(probabilities).all():
        raise ValueError("Invalid probabilities")
    return probabilities


def model_inventory():
    return {str(path.relative_to(MODEL_DIR)): sha256_file(path)
            for path in sorted(MODEL_DIR.rglob("*")) if path.is_file() and path != LOCK_PATH}


def load_and_verify_lock():
    if not LOCK_PATH.is_file():
        raise FileNotFoundError("Complete A validation/calibration and freeze the model first")
    locked = json.loads(LOCK_PATH.read_text(encoding="utf-8"))
    if locked["protocol"] != protocol or locked["model_files_sha256"] != model_inventory():
        raise ValueError("Locked model/protocol changed: refuse final test inference")
    return locked


protocol, resume_checkpoint = initialize_run()
SCORER = OUTPUT_DIR / "scorer-subtask-1.py"
LABEL_FILE = OUTPUT_DIR / "labels.txt"
if RUN_TRAINING:
    fit_baseline(resume_checkpoint)
else:
    load_and_verify_lock()
    print("Loaded previously completed run; training and threshold search will be skipped.")


## 4. Hiệu chỉnh ngưỡng chỉ bằng validation A và khóa bundle

Tìm ngưỡng chung trong lưới rồi coordinate search tối đa ba vòng cho từng nhãn, mục tiêu micro-F1
có None. Nhãn không có mẫu dương trong validation giữ ngưỡng chung. Không dùng Top-1 fallback.
Báo cả ngưỡng 0,50 và ngưỡng đã chọn. Điểm validation sau hiệu chỉnh có thiên lệch chọn tham số;
điểm B là kết quả kiểm nghiệm cục bộ cuối. Model đã lưu được nạp lại trước khi hiệu chỉnh.


In [ ]:
if RUN_TRAINING:
    validation_probabilities = predict_saved_model(val_df)
    label_thresholds, calibration_report = calibrate_thresholds(
        val_df[LABELS].to_numpy(dtype=np.int8), validation_probabilities, THRESHOLD_GRID)
    for suffix, thresholds in [("at_0_5", 0.50), ("calibrated", label_thresholds)]:
        validation_metrics = save_evaluation(val_df, validation_probabilities, thresholds,
            OUTPUT_DIR, "validation_A_" + suffix, SCORER, LABEL_FILE, training_languages)
    (OUTPUT_DIR / "calibration_A.json").write_text(json.dumps(calibration_report, indent=2), encoding="utf-8")
    pd.DataFrame({"label": LABELS, "threshold": label_thresholds}).to_csv(
        OUTPUT_DIR / "label_thresholds.csv", index=False)
    locked = {"protocol": protocol, "labels": LABELS, "thresholds": label_thresholds.tolist(),
              "calibrated_on": "validation_A_only", "input_format": "title_then_target_xlmr_pair",
              "model_files_sha256": model_inventory(), "calibration": calibration_report}
    # Exclusive creation: do not overwrite a prior completed experiment.
    with LOCK_PATH.open("x", encoding="utf-8") as handle:
        json.dump(locked, handle, ensure_ascii=False, indent=2)
else:
    locked = load_and_verify_lock()
    label_thresholds = np.asarray(locked["thresholds"], dtype=np.float64)
print("Model and validation-only thresholds are frozen:", LOCK_PATH)
display(pd.DataFrame({"label": LABELS, "threshold": label_thresholds}))


## 5. Suy luận B sau khi khóa model/ngưỡng

Reader ở bước này chỉ lấy ID, offset, đoạn và tiêu đề; **không parse nhãn B**, không tạo cột gold giả.
Kiểm tra không trùng bài/nội dung giữa A và B. Giữ nguyên mọi hàng B kể cả hai bài trùng nhau nội bộ.
Lưu xác suất, khóa hàng và TSV dự đoán trước khi mở gold. Nếu tiếp tục nghiên cứu, không dùng điểm B
để chọn lặp lại siêu tham số rồi vẫn coi nó là test độc lập.


In [ ]:
locked = load_and_verify_lock()
label_thresholds = np.asarray(locked["thresholds"], dtype=np.float64)
TEST_DATA_DIR = locate_package(prepare_dataset(TEST_DATA_PATH, WORK_DIR / "datasets_cache"), "B")
test_inputs = read_benchmark_package(TEST_DATA_DIR, with_gold=False, source="B")
if set(LABELS + ["frames", "is_none"]) & set(test_inputs.columns):
    raise AssertionError("Test inference must not receive gold columns")
assert_no_test_overlap(development_df, test_inputs)
test_input_sha256 = frame_fingerprint(test_inputs, with_gold=False)
receipt_path = OUTPUT_DIR / "test_B_prediction_receipt.json"
prediction_path = OUTPUT_DIR / "pred_semeval2023_test.tsv"
probability_path = OUTPUT_DIR / "test_B_inference_probabilities.npy"
if receipt_path.is_file():
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    if (receipt["input_sha256"] != test_input_sha256
            or receipt["ordered_keys_sha256"] != ordered_key_fingerprint(test_inputs)
            or receipt["lock_sha256"] != sha256_file(LOCK_PATH)
            or receipt["prediction_sha256"] != sha256_file(prediction_path)
            or receipt["probability_sha256"] != sha256_file(probability_path)):
        raise ValueError("Existing B inference artifacts changed. Use a new run for another experiment.")
    test_probabilities = np.load(probability_path, allow_pickle=False)
    print("Reusing final predictions from this identical locked run.")
else:
    test_probabilities = predict_saved_model(test_inputs)
    np.save(probability_path, test_probabilities)
    write_predictions_tsv(test_inputs, apply_thresholds(test_probabilities, label_thresholds), prediction_path)
    test_inputs[KEY_COLUMNS + ["language", "article_sha256"]].to_csv(
        OUTPUT_DIR / "test_B_prediction_keys.csv", index=False)
    receipt = {"input_sha256": test_input_sha256,
               "ordered_keys_sha256": ordered_key_fingerprint(test_inputs),
               "lock_sha256": sha256_file(LOCK_PATH),
               "prediction_sha256": sha256_file(prediction_path),
               "probability_sha256": sha256_file(probability_path), "rows": len(test_inputs),
               "gold_parsed_before_prediction": False}
    receipt_path.write_text(json.dumps(receipt, indent=2), encoding="utf-8")
print(f"Saved {len(test_inputs):,} final predictions:", prediction_path)
display(data_summary(test_inputs))


## 6. Mở gold B để tính metrics và đối chiếu scorer

Đây là bước đánh giá sau inference, không train hay tìm thêm ngưỡng. Xuất:
micro/macro P/R/F1, 14-frame F1, sample-F1, subset accuracy, Hamming loss,
average precision, per-label/per-language và nhóm ngôn ngữ đã/chưa có trong train A.
Scorer chạy bằng subprocess và phải khớp F1 nội bộ. Gold TSV được ghép từ B theo đúng các khóa.
`official_*` nghĩa là **khớp scorer ST1 2027 nhúng trong notebook**, không phải điểm official test 2023.
Scorer dùng `zero_division=1`: nếu cả gold/prediction đều không có None, F1 của lớp None bằng 1.
Vì thế cần đọc riêng macro-F1 14 frame (`zero_division=0`) và macro-F1 15 lớp của scorer.


In [ ]:
if not receipt_path.is_file() or sha256_file(prediction_path) != receipt["prediction_sha256"]:
    raise RuntimeError("Write and verify predictions BEFORE reading B gold")
test_gold = read_benchmark_package(TEST_DATA_DIR, with_gold=True, source="B")
if frame_fingerprint(test_gold, with_gold=False) != test_input_sha256:
    raise ValueError("B inputs changed between prediction and scoring")
# Align by keys, not by an assumption about file/row order.
indexed_gold = test_gold.set_index(KEY_COLUMNS, verify_integrity=True)
key_index = pd.MultiIndex.from_frame(test_inputs[KEY_COLUMNS])
test_gold = indexed_gold.loc[key_index].reset_index()
test_metrics = save_evaluation(test_gold, test_probabilities, label_thresholds,
    OUTPUT_DIR, "test_B", SCORER, LABEL_FILE, training_languages)
if (OUTPUT_DIR / "test_B_predictions.tsv").read_bytes() != prediction_path.read_bytes():
    raise AssertionError("Scored predictions differ from frozen inference TSV")
data_summary(test_gold).to_csv(OUTPUT_DIR / "package_B_summary.csv", index=False)
display(pd.Series(test_metrics, name="test_B").to_frame())
display(pd.read_csv(OUTPUT_DIR / "test_B_per_label_metrics.csv", keep_default_na=False))
display(pd.read_csv(OUTPUT_DIR / "test_B_per_language_metrics.csv"))
print("Local paragraph A→B benchmark; NOT the original document-level 2023 leaderboard.")


## 7. Biểu đồ và gói kết quả

Biểu đồ mô tả kết quả cuối, không dùng để chọn lại ngưỡng B. Model đầy đủ nằm trong `/kaggle/working/` ở
thư mục run; ZIP gọn dưới đây gồm log, metrics, TSV, xác suất, manifests và lock, không chứa trọng số lớn.


In [ ]:
import matplotlib.pyplot as plt

label_results = pd.read_csv(OUTPUT_DIR / "test_B_per_label_metrics.csv", keep_default_na=False)
language_results = pd.read_csv(OUTPUT_DIR / "test_B_per_language_metrics.csv")
# Column names are fixed by the embedded evaluation helper.
frames = label_results.loc[label_results["label"].ne("None")].sort_values("f1")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(frames["label"], frames["f1"])
axes[0].set(xlim=(0, 1), xlabel="F1", title="B: F1 by frame (14 labels)")
axes[1].bar(language_results["language"], language_results["official_micro_f1"])
axes[1].set(ylim=(0, 1), ylabel="Micro-F1 (2027 scorer)", title="B: F1 by language")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "test_B_metrics.png", dpi=160, bbox_inches="tight")
plt.show()

archive_path = RUN_DIR / "benchmark_results.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, str(path.relative_to(RUN_DIR)))
    archive.write(RUN_DIR / "protocol.json", "protocol.json")
    archive.write(LOCK_PATH, "model/benchmark_lock.json")
print("Model/tokenizer/lock:", MODEL_DIR)
print("Metrics and predictions:", OUTPUT_DIR)
print("Results ZIP (without large model weights):", archive_path)
print("Kaggle: Save Version, then download the model folder or ZIP from Output/Files.")


## Cách dùng làm mốc đối chứng nghiên cứu

Giữ nguyên A/B, split manifest, seed, scorer và token budget khi so thuật toán tiếp theo.
Lựa chọn mô hình trên validation A; hạn chế nhìn B nhiều lần. Báo rõ từng ngôn ngữ và chi phí train.
Calibration ở A có thể kém tổng quát hóa sang EN/ES/GR/KA do A không có các ngôn ngữ đó.
Không có gold None khiến kết quả chưa đo được năng lực loại trừ frame trên negative thật.

Muốn tái hiện **SemEval 2023 ST2 gốc** cần lấy dữ liệu/nhãn **cấp bài**, split và scorer gốc.
Khi chưa có bộ đó, không chuyển điểm A→B này thành tuyên bố vượt các hệ thống năm 2023.
